In [1]:
from datetime import datetime, timedelta
import pandas as pd
import requests


def fetch_indonesia_weather_dataset():
  # Define major Indonesian cities with their geographical coordinates
  cities = [
      {"city": "Jakarta", "latitude": -6.2088, "longitude": 106.8456},
      {"city": "Surabaya", "latitude": -7.2575, "longitude": 112.7521},
      {"city": "Bandung", "latitude": -6.9175, "longitude": 107.6191},
      {"city": "Medan", "latitude": 3.5952, "longitude": 98.6722},
      {"city": "Denpasar", "latitude": -8.6705, "longitude": 115.2126},
      {"city": "Semarang", "latitude": -6.9666, "longitude": 110.4194},
      {"city": "Makassar", "latitude": -5.1477, "longitude": 119.4327},
      {"city": "Yogyakarta", "latitude": -7.7956, "longitude": 110.3695},
      {"city": "Palembang", "latitude": -2.9909, "longitude": 104.7565},
      {"city": "Balikpapan", "latitude": -1.2654, "longitude": 116.8312},
  ]

  # Define date range for the past 7 days up to yesterday (Archive API support)
  end_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
  start_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

  all_records = []
  print(
      f"Fetching historical weather data ({start_date} to {end_date}) for"
      " Indonesian cities..."
  )

  # Define meteorological variables to retrieve from Open-Meteo Archive API
  weather_variables = [
      "temperature_2m",
      "relative_humidity_2m",
      "precipitation",
      "wind_speed_10m",
  ]

  for location in cities:
    # Open-Meteo Historical Weather API endpoint
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(weather_variables),
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
      data = response.json()
      hourly_data = data.get("hourly", {})
      timestamps = hourly_data.get("time", [])

      # Map each weather variable safely into lists
      metric_values = {
          v: hourly_data.get(v, [None] * len(timestamps))
          for v in weather_variables
      }

      # Loop through each hourly record and structure it into dictionaries
      for i, timestamp in enumerate(timestamps):
        record = {
            "city": location["city"],
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "time": timestamp,
        }
        for v in weather_variables:
          record[v] = (
              metric_values[v][i] if i < len(metric_values[v]) else None
          )

        all_records.append(record)
    else:
      print(f"Failed to fetch weather data for {location['city']}")

  # Convert collected records to a Pandas DataFrame and export as CSV
  if all_records:
    df = pd.DataFrame(all_records)
    filename = "indonesia_weather_timeseries.csv"
    df.to_csv(filename, index=False)

    print(
        f"\nSuccess! Total records collected: {len(df)} rows.\nDataset saved"
        f" as: '{filename}'"
    )

    # Automatically trigger file download if running inside Google Colab
    try:
      from google.colab import files

      files.download(filename)
    except ImportError:
      pass
  else:
    print("No weather data collected.")


if __name__ == "__main__":
  fetch_indonesia_weather_dataset()

Fetching historical weather data (2026-10-08 to 2026-10-14) for Indonesian cities...

Success! Total records collected: 1680 rows.
Dataset saved as: 'indonesia_weather_timeseries.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>